# 🏆 Buổi 8/9 — Submission: Tạo File & Nộp Bài Kaggle

### Bước cuối cùng: biến model thành file CSV rồi nộp lên Kaggle!

---

> **📍 Bạn đang ở đây trong lộ trình 9 buổi:**
> ```
> Buổi 1 ✅ Giới thiệu dự án & lộ trình
> Buổi 2 ✅ Setup môi trường & khám phá SalePrice
> Buổi 3 ✅ EDA — Phân tích missing, tương quan, outliers
> Buổi 4 ✅ Data Cleaning & Preprocessing
> Buổi 5 ✅ Feature Engineering
> Buổi 6 ✅ Model Training (Linear + Tree + Stacking)
> Buổi 7 ✅ Model Evaluation (Đánh giá toàn diện)
> Buổi 8 🔄 Submit Kaggle (tạo file & nộp bài)  ← BẠN ĐANG Ở ĐÂY
> Buổi 9 ⏳ Quiz Tốt Nghiệp (20 câu ôn tập)
> ```

---

### 📋 Nội Dung Buổi 8

| # | Task | Nội dung |
|---|------|----------|
| 1 | ♻️ Rebuild Pipeline | Load data & xử lý lại từ đầu (self-contained) |
| 2 | 🎯 Train Final Models | Train tất cả models trên **toàn bộ** training data |
| 3 | 🤝 Best Ensemble | Chọn và kết hợp models tốt nhất |
| 4 | 📊 Phân Tích Dự Đoán | Kiểm tra phân phối, phát hiện bất thường |
| 5 | 📁 Tạo File CSV | Chuyển đổi log → giá thực, xuất file submission |
| 6 | ✅ Validate Submission | Kiểm tra format trước khi nộp |
| 7 | 🚀 Nộp Kaggle | Hướng dẫn upload và đọc kết quả |

### 🎯 Mục tiêu
- Tạo được file `submission.csv` đúng format Kaggle
- Đạt **Public Leaderboard RMSE < 0.13**
- Hiểu quy trình **end-to-end**: từ raw data → submission

> 💡 **Notebook này hoàn toàn self-contained** — chạy từ đầu đến cuối mà không cần chạy các buổi trước!


---

## ♻️ Task 1 — Rebuild Pipeline (Self-contained)

Notebook này **tự load & xử lý data từ đầu** để đảm bảo chạy độc lập.  
Toàn bộ pipeline từ Buổi 4–6 được đóng gói vào đây.


In [ ]:
# ============================================================
# ♻️ TASK 1 — Import & Rebuild Full Pipeline
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV, LassoCV, Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_squared_error

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
    print("✅ XGBoost sẵn sàng")
except ImportError:
    HAVE_XGB = False
    print("⚠️  XGBoost chưa cài — pip install xgboost")

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── 1.1 Load data ────────────────────────────────────────
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

test_id  = test_df['Id'].copy()          # giữ lại Id để tạo submission
y        = np.log1p(train_df['SalePrice'])

train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

ntrain = len(train_df)
ntest  = len(test_df)
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

print(f"   train: {ntrain} rows | test: {ntest} rows | features: {all_data.shape[1]}")

# ── 1.2 Fill missing ──────────────────────────────────────
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
mode_cols = ['MSZoning','Utilities','Functional','Electrical',
             'KitchenQual','Exterior1st','Exterior2nd','SaleType']

for col in none_cols: all_data[col] = all_data[col].fillna('None')
for col in zero_cols: all_data[col] = all_data[col].fillna(0)
all_data['LotFrontage'] = (all_data.groupby('Neighborhood')['LotFrontage']
                           .transform(lambda x: x.fillna(x.median())))
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── 1.3 Xoá outlier ──────────────────────────────────────
outlier_mask = (all_data['GrLivArea'][:ntrain] > 4000) & (np.expm1(y) < 300_000)
outlier_idx  = outlier_mask[outlier_mask].index.tolist()
all_data.drop(index=outlier_idx, inplace=True)
all_data.reset_index(drop=True, inplace=True)
y.drop(index=outlier_idx, inplace=True)
y.reset_index(drop=True, inplace=True)
ntrain -= len(outlier_idx)
print(f"   Xóa {len(outlier_idx)} outlier → train còn {ntrain} rows")

# ── 1.4 Feature Engineering ───────────────────────────────
all_data['TotalSF']      = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] +
                             all_data['EnclosedPorch'] + all_data['ScreenPorch'] + all_data['WoodDeckSF'])
all_data['TotalBath']    = (all_data['FullBath'] + 0.5*all_data['HalfBath'] +
                             all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath'])
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['HasGarage']    = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt']      = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HouseAge']     = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']     = all_data['YrSold'] - all_data['YearRemodAdd']

for feat in ['OverallQual', 'GrLivArea', 'TotalSF']:
    all_data[f'{feat}_sq']   = all_data[feat] ** 2
    all_data[f'{feat}_sqrt'] = np.sqrt(all_data[feat])

for feat in ['GrLivArea', 'TotalSF', 'LotArea']:
    all_data[f'{feat}_log'] = np.log1p(all_data[feat])

# ── 1.5 Ordinal Encoding ──────────────────────────────────
ordinal_mappings = {
    'ExterQual':    ['None','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['None','Po','Fa','TA','Gd','Ex'],
    'KitchenQual':  ['None','Po','Fa','TA','Gd','Ex'],
    'FireplaceQu':  ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'GarageQual':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond':   ['None','Po','Fa','TA','Gd','Ex'],
    'PoolQC':       ['None','Fa','TA','Gd','Ex'],
    'Fence':        ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'Functional':   ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'LandSlope':    ['Sev','Mod','Gtl'],
    'LotShape':     ['IR3','IR2','IR1','Reg'],
    'PavedDrive':   ['N','P','Y'],
}
for col, order in ordinal_mappings.items():
    mapping = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping).fillna(0).astype(int)

# ── 1.6 One-Hot + Variance Filter + Scale ────────────────
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
all_data_enc = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)

X_train_raw = all_data_enc[:ntrain].copy()
X_test_raw  = all_data_enc[ntrain:].copy()
X_test_raw.reset_index(drop=True, inplace=True)

selector      = VarianceThreshold(threshold=0.01)
selected_mask = selector.fit(X_train_raw).get_support()
X_train_sel   = X_train_raw.loc[:, selected_mask]
X_test_sel    = X_test_raw.loc[:, selected_mask]

scaler  = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_sel), columns=X_train_sel.columns)
X_test  = pd.DataFrame(scaler.transform(X_test_sel),      columns=X_test_sel.columns)

# ── 1.7 CV helper ─────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model):
    scores = cross_val_score(model, X_train, y,
                             scoring='neg_root_mean_squared_error', cv=kf)
    return float(-scores.mean())

print(f"\n✅ Pipeline sẵn sàng!")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y       : {y.shape}  (log1p SalePrice)")


**Expected Output:**

![img8-1](images/img_buoi8/img8-1.png)

---

## 🎯 Task 2 — Train Final Models (trên toàn bộ training data)

Ở buổi 6 ta đã **cross-validate** để chọn model.  
Bây giờ ta **train lại lần cuối** trên **100% training data** để có model mạnh nhất trước khi dự đoán test set.

> **Tại sao train lại?** CV chỉ train trên ~80% data mỗi fold — train lại trên 100% giúp model học được nhiều pattern hơn!


In [ ]:
# ============================================================
# 🎯 TASK 2 — Train All Final Models + CV RMSE
# ============================================================
print("=" * 55)
print("  Training models & đánh giá CV RMSE...")
print("=" * 55)

# ── 2.1 Ridge ─────────────────────────────────────────────
ridge = RidgeCV(alphas=[1, 10, 50, 100, 300, 500])
ridge.fit(X_train, y)
ridge_rmse = rmse_cv(ridge)
print(f"\n[1/5] Ridge        CV RMSE = {ridge_rmse:.4f}  (alpha={ridge.alpha_:.0f})")

# ── 2.2 Lasso ─────────────────────────────────────────────
lasso = LassoCV(alphas=[0.0001, 0.0003, 0.001, 0.003, 0.01], cv=kf, max_iter=5000)
lasso.fit(X_train, y)
lasso_rmse = rmse_cv(lasso)
nonzero    = int((lasso.coef_ != 0).sum())
print(f"[2/5] Lasso        CV RMSE = {lasso_rmse:.4f}  ({nonzero} features)")

# ── 2.3 Random Forest (tune nhẹ) ──────────────────────────
rf = RandomForestRegressor(n_estimators=300, max_features='sqrt',
                           min_samples_leaf=1, random_state=42, n_jobs=-1)
rf.fit(X_train, y)
rf_rmse = rmse_cv(rf)
print(f"[3/5] RandomForest CV RMSE = {rf_rmse:.4f}")

# ── 2.4 XGBoost ───────────────────────────────────────────
if HAVE_XGB:
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.05,
                       max_depth=4, subsample=0.8, colsample_bytree=0.8,
                       reg_alpha=0.01, reg_lambda=1.0,
                       random_state=42, verbosity=0)
    xgb.fit(X_train, y)
    xgb_rmse = rmse_cv(xgb)
    print(f"[4/5] XGBoost      CV RMSE = {xgb_rmse:.4f}")
else:
    xgb_rmse = None

# ── 2.5 Stacking ──────────────────────────────────────────
base_estimators = [
    ('ridge', Ridge(alpha=50)),
    ('rf',    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
]
if HAVE_XGB:
    base_estimators.append(
        ('xgb', XGBRegressor(n_estimators=300, learning_rate=0.05,
                              max_depth=4, subsample=0.8, random_state=42, verbosity=0))
    )

stacking = StackingRegressor(estimators=base_estimators,
                              final_estimator=Ridge(alpha=10), cv=5)
stacking_rmse = rmse_cv(stacking)
stacking.fit(X_train, y)
print(f"[5/5] Stacking     CV RMSE = {stacking_rmse:.4f}")

# ── Bảng tổng kết ─────────────────────────────────────────
print("\n" + "=" * 55)
print("  📊 BẢNG TỔNG KẾT CV RMSE")
print("=" * 55)
results = {
    'Ridge':         ridge_rmse,
    'Lasso':         lasso_rmse,
    'Random Forest': rf_rmse,
    'Stacking':      stacking_rmse,
}
if HAVE_XGB:
    results['XGBoost'] = xgb_rmse

best_name = min(results, key=results.get)
for name, score in sorted(results.items(), key=lambda x: x[1]):
    star = " ★ BEST" if name == best_name else ""
    print(f"  {name:<16} {score:.4f}{star}")
print("=" * 55)


**Expected Output:**

![img8-2](images/img_buoi8/img8-2.png)

---

## 🤝 Task 3 — Weighted Ensemble (Kết hợp tốt nhất)

Thay vì chọn 1 model duy nhất, ta **kết hợp có trọng số** các models tốt nhất.  
Model nào CV RMSE thấp hơn → được gán **trọng số cao hơn**.

### Công thức Weighted Blend:

$$\hat{y}_{final} = w_1 \cdot \hat{y}_{ridge} + w_2 \cdot \hat{y}_{lasso} + w_3 \cdot \hat{y}_{xgb} + w_4 \cdot \hat{y}_{stacking}$$

Trong đó $\sum w_i = 1$ và trọng số tỉ lệ nghịch với CV RMSE của từng model.


In [ ]:
# ============================================================
# 🤝 TASK 3 — Weighted Ensemble (dự đoán trên test set)
# ============================================================

# ── 3.1 Dự đoán log(SalePrice) trên test set ─────────────
pred_ridge    = ridge.predict(X_test)
pred_lasso    = lasso.predict(X_test)
pred_rf       = rf.predict(X_test)
pred_stacking = stacking.predict(X_test)

# ── 3.2 Tính trọng số: w ∝ 1/RMSE (model tốt → weight lớn)
rmse_list  = [ridge_rmse, lasso_rmse, rf_rmse, stacking_rmse]
pred_list  = [pred_ridge, pred_lasso, pred_rf, pred_stacking]
name_list  = ['Ridge', 'Lasso', 'RF', 'Stacking']

if HAVE_XGB:
    pred_xgb  = xgb.predict(X_test)
    rmse_list.append(xgb_rmse)
    pred_list.append(pred_xgb)
    name_list.append('XGBoost')

inv_rmse = np.array([1.0 / r for r in rmse_list])
weights  = inv_rmse / inv_rmse.sum()   # chuẩn hóa tổng = 1

print("📐 Trọng số từng model (tổng = 1.0):")
for nm, w, r in sorted(zip(name_list, weights, rmse_list), key=lambda x: -x[1]):
    bar = '█' * int(w * 40)
    print(f"  {nm:<14} weight={w:.3f}  {bar}  (CV RMSE={r:.4f})")

# ── 3.3 Blend: trung bình có trọng số ────────────────────
pred_blend_log = sum(w * p for w, p in zip(weights, pred_list))

# ── 3.4 Chuyển ngược log → giá gốc ──────────────────────
# y = log1p(SalePrice)  →  SalePrice = expm1(y)
pred_final = np.expm1(pred_blend_log)

print(f"\n✅ Đã tạo xong dự đoán cuối!")
print(f"   Số dự đoán  : {len(pred_final)}")
print(f"   Giá thấp nhất  : ${pred_final.min():>12,.0f}")
print(f"   Giá cao nhất   : ${pred_final.max():>12,.0f}")
print(f"   Giá trung bình : ${pred_final.mean():>12,.0f}")
print(f"   Giá median     : ${np.median(pred_final):>12,.0f}")


**Expected Output:**

![img8-3](images/img_buoi8/img8-3.png)

---

## 📊 Task 4 — Phân Tích Dự Đoán (Kiểm Tra Trước Khi Submit)

Trước khi nộp, cần **kiểm tra kỹ** dự đoán:
- Phân phối có hợp lý không? (không có giá âm, không có giá bất thường)
- So sánh với phân phối training data
- Phát hiện outlier trong dự đoán


In [ ]:
# ============================================================
# 📊 TASK 4 — Phân Tích & Kiểm Tra Dự Đoán
# ============================================================
train_prices = np.expm1(y)   # giá gốc của training set

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Panel 1: So sánh phân phối Train vs Test predictions ──
ax = axes[0, 0]
ax.hist(train_prices / 1000, bins=50, alpha=0.6, color='#2980b9',
        label=f'Train (n={len(train_prices)})', density=True)
ax.hist(pred_final / 1000,   bins=50, alpha=0.6, color='#e67e22',
        label=f'Test predictions (n={len(pred_final)})', density=True)
ax.set_xlabel('SalePrice (nghìn USD)', fontsize=10)
ax.set_ylabel('Mật độ', fontsize=10)
ax.set_title('Phân phối: Train vs Test Predictions\n(nên có hình dạng tương đồng!)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 2: Log-scale distribution ───────────────────────
ax = axes[0, 1]
ax.hist(y,                   bins=50, alpha=0.6, color='#2980b9', label='Train log(SalePrice)', density=True)
ax.hist(pred_blend_log,      bins=50, alpha=0.6, color='#e67e22', label='Test log predictions', density=True)
ax.set_xlabel('log(SalePrice)', fontsize=10)
ax.set_ylabel('Mật độ', fontsize=10)
ax.set_title('Phân phối Log Scale\n(kiểm tra xem có bình thường không)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 3: Boxplot từng model và blend ──────────────────
ax = axes[1, 0]
box_data  = [np.expm1(p) / 1000 for p in pred_list]
box_data.append(pred_final / 1000)
box_names = name_list + ['★ Blend']
bp = ax.boxplot(box_data, labels=box_names, patch_artist=True, notch=False)
colors_box = ['#3498db', '#e74c3c', '#27ae60', '#9b59b6', '#e67e22', '#8e44ad']
for patch, col in zip(bp['boxes'], colors_box):
    patch.set_facecolor(col)
    patch.set_alpha(0.6)
ax.set_ylabel('SalePrice (nghìn USD)', fontsize=10)
ax.set_title('Boxplot Dự Đoán Từng Model\n(★ Blend = kết hợp cuối)', fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 4: Histogram giá dự đoán với thống kê ───────────
ax = axes[1, 1]
n_bins = 40
counts, bin_edges, patches = ax.hist(pred_final / 1000, bins=n_bins,
                                      color='#8e44ad', alpha=0.75, edgecolor='white')
# Tô màu outlier (quá thấp / quá cao)
q1, q99 = np.percentile(pred_final, 1), np.percentile(pred_final, 99)
for patch, left in zip(patches, bin_edges[:-1]):
    val = left * 1000
    if val < q1 or val > q99:
        patch.set_facecolor('#e74c3c')
        patch.set_alpha(0.9)

ax.axvline(np.median(pred_final) / 1000, color='#2ecc71', lw=2,
           linestyle='--', label=f'Median: ${np.median(pred_final)/1000:.0f}K')
ax.axvline(pred_final.mean() / 1000, color='#f39c12', lw=2,
           linestyle='--', label=f'Mean: ${pred_final.mean()/1000:.0f}K')
ax.axvline(q1 / 1000, color='#e74c3c', lw=1.5, linestyle=':',
           label=f'P1: ${q1/1000:.0f}K (cảnh báo)')
ax.axvline(q99 / 1000, color='#e74c3c', lw=1.5, linestyle=':',
           label=f'P99: ${q99/1000:.0f}K (cảnh báo)')
ax.set_xlabel('SalePrice dự đoán (nghìn USD)', fontsize=10)
ax.set_ylabel('Số lượng nhà', fontsize=10)
ax.set_title('Phân phối Giá Dự Đoán\n(Đỏ = ngoài khoảng P1–P99)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Kiểm Tra Chất Lượng Dự Đoán Trước Khi Submit',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Cảnh báo tự động ─────────────────────────────────────
print("\n🔍 Kiểm tra tự động:")
n_neg    = (pred_final <= 0).sum()
n_low    = (pred_final < 50_000).sum()
n_high   = (pred_final > 800_000).sum()
n_nan    = np.isnan(pred_final).sum()
print(f"  Giá âm hoặc = 0         : {n_neg}  {'⚠️  CÓ VẤN ĐỀ!' if n_neg > 0 else '✅ OK'}")
print(f"  Giá < $50,000            : {n_low}  {'⚠️  Nghi ngờ!' if n_low > 5 else '✅ OK'}")
print(f"  Giá > $800,000           : {n_high}  {'⚠️  Nghi ngờ!' if n_high > 5 else '✅ OK'}")
print(f"  Giá trị NaN              : {n_nan}  {'⚠️  CÓ VẤN ĐỀ!' if n_nan > 0 else '✅ OK'}")
print(f"  Số dự đoán = {len(test_id)} (ntest)  : {'✅ Đủ!' if len(pred_final) == len(test_id) else '⚠️  Thiếu!'}")


**Expected Output:**

![img8-4](images/img_buoi8/img8-4.png)
![img8-5](images/img_buoi8/img8-5.png)

### 📖 Cách Đọc 4 Biểu Đồ Trên

---

**① Phân phối: Train vs Test Predictions** *(góc trên trái)*

So sánh hình dạng phân phối giá giữa tập train và tập test được dự đoán.  
- ✅ **Tốt:** Hai đường gần trùng nhau → model dự đoán ra phân phối hợp lý, không bị lệch.  
- ⚠️ **Xấu:** Đường cam (test) dịch sang trái hoặc phải quá nhiều → model đang underestimate hoặc overestimate giá nhà.

---

**② Phân phối Log Scale** *(góc trên phải)*

Tương tự biểu đồ ①, nhưng nhìn ở **log(SalePrice)** — đây là không gian mà model thực sự học.  
- ✅ **Tốt:** Hai đường xấp xỉ nhau, đều có dạng hình chuông cân đối.  
- ⚠️ **Xấu:** Đường test bị lệch, hoặc có đuôi dài bất thường → cần kiểm tra lại pipeline xử lý dữ liệu.

---

**③ Boxplot từng Model** *(góc dưới trái)*

So sánh phân phối dự đoán của **từng model riêng lẻ** và **Blend (★)**.  
- **Hộp (box):** 50% giá trị trung tâm — hộp hẹp = model ổn định, hộp rộng = model dao động nhiều.  
- **Đường giữa hộp:** Median dự đoán.  
- **Râu (whiskers):** Khoảng giá trị bình thường.  
- **Chấm ngoài râu:** Outlier — nhà được dự đoán giá bất thường.  
- ✅ **Tốt:** Blend (★) có median gần với các model, không quá cao hoặc quá thấp.

---

**④ Phân phối Giá Dự Đoán** *(góc dưới phải)*

Histogram toàn bộ dự đoán của Blend cuối cùng.  
- **Màu tím:** Vùng giá bình thường (trong khoảng P1–P99).  
- **Màu đỏ:** Vùng bất thường — ít hơn 1% hoặc nhiều hơn 99% → cần để ý.  
- **Đường xanh lá (Median):** Giá giữa — ít bị ảnh hưởng bởi outlier.  
- **Đường cam (Mean):** Giá trung bình — nếu Mean >> Median thì có outlier giá cao kéo lên.  
- ✅ **Tốt:** Phần lớn dự đoán nằm trong vùng tím, Mean và Median không chênh nhau quá nhiều (< 10%).


---

## 📁 Task 5 — Tạo File submission.csv

Format Kaggle yêu cầu đúng 2 cột: `Id` và `SalePrice`.

```
Id,SalePrice
1461,169277.05
1462,187758.39
...
```

> ⚠️ **Lưu ý quan trọng:** Giá phải là **giá gốc** (USD), KHÔNG phải log!  
> Ta đã dùng `np.expm1()` để chuyển ngược từ log → giá gốc ở bước trên.


In [ ]:
# ============================================================
# 📁 TASK 5 — Tạo File submission.csv
# ============================================================

submission = pd.DataFrame({
    'Id':        test_id.values,
    'SalePrice': pred_final
})

# Đảm bảo không có giá âm (clip về mức tối thiểu an toàn)
min_price = 50_000
submission['SalePrice'] = submission['SalePrice'].clip(lower=min_price)

# Xuất ra file CSV
output_path = 'data/submission.csv'
submission.to_csv(output_path, index=False)

print(f"✅ Đã lưu file: {output_path}")
print(f"\n📋 5 dòng đầu tiên:")
print(submission.head().to_string(index=False))

print(f"\n📋 5 dòng cuối cùng:")
print(submission.tail().to_string(index=False))

print(f"\n📊 Thống kê SalePrice:")
print(f"  Count  : {len(submission)}")
print(f"  Min    : ${submission['SalePrice'].min():>12,.2f}")
print(f"  Max    : ${submission['SalePrice'].max():>12,.2f}")
print(f"  Mean   : ${submission['SalePrice'].mean():>12,.2f}")
print(f"  Median : ${submission['SalePrice'].median():>12,.2f}")
print(f"  Std    : ${submission['SalePrice'].std():>12,.2f}")


**Expected Output:**

![img8-6](images/img_buoi8/img8-6.png)

---

## ✅ Task 6 — Validate Submission

Kiểm tra file đã tạo có đúng format Kaggle chưa trước khi upload.


In [ ]:
# ============================================================
# ✅ TASK 6 — Validate Submission Format
# ============================================================
import os

# ── Đọc lại file vừa lưu (kiểm tra độc lập) ─────────────
sub_check = pd.read_csv(output_path)
sample    = pd.read_csv('data/sample_submission.csv')

print("=" * 55)
print("  KIỂM TRA FORMAT SUBMISSION")
print("=" * 55)

checks = {
    "File tồn tại"              : os.path.exists(output_path),
    "Đúng 2 cột (Id, SalePrice)": list(sub_check.columns) == ['Id', 'SalePrice'],
    f"Đúng {len(sample)} dòng"  : len(sub_check) == len(sample),
    "Không có NaN"              : sub_check.isnull().sum().sum() == 0,
    "Không có giá âm"           : (sub_check['SalePrice'] > 0).all(),
    "Id khớp sample_submission" : (sub_check['Id'].values == sample['Id'].values).all(),
    "Kiểu Id là số nguyên"      : sub_check['Id'].dtype in [np.int64, np.int32],
    "Kiểu SalePrice là số thực" : sub_check['SalePrice'].dtype in [np.float64, np.float32],
}

all_pass = True
for check_name, result in checks.items():
    icon = "✅" if result else "❌"
    print(f"  {icon} {check_name}")
    if not result:
        all_pass = False

print("=" * 55)
if all_pass:
    file_kb = os.path.getsize(output_path) / 1024
    print(f"  🎉 TẤT CẢ KIỂM TRA ĐỀU PASS!")
    print(f"  📁 File size: {file_kb:.1f} KB → {output_path}")
else:
    print("  ⚠️  Có lỗi — kiểm tra lại trước khi submit!")
print("=" * 55)

# ── So sánh trực quan với sample_submission ───────────────
print(f"\n📋 Submission của bạn vs sample_submission:")
compare = pd.DataFrame({
    'Id':              sub_check['Id'],
    'Predicted':       sub_check['SalePrice'].round(2),
    'Sample_baseline': sample['SalePrice'].round(2),
    'Difference':      (sub_check['SalePrice'] - sample['SalePrice']).round(2),
})
print(compare.head(10).to_string(index=False))


**Expected Output:**

![img8-7](images/img_buoi8/img8-7.png)

---

## 🚀 Task 7 — Nộp Bài Kaggle & Đọc Kết Quả

### Cách 1: Upload thủ công (khuyến nghị cho người mới)

1. Truy cập: **[https://www.kaggle.com/c/house-prices-advanced-regression-techniques/submit](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/submit)**
2. Nhấn **"Upload Submission File"** → chọn file `submission.csv`
3. Điền mô tả ngắn (ví dụ: `"Weighted Ensemble: Ridge+Lasso+RF+XGBoost+Stacking"`)
4. Nhấn **"Make Submission"**
5. Chờ ~1 phút → xem kết quả **Public Score** trên leaderboard

### Cách 2: Dùng Kaggle API (nâng cao)

```bash
# Cài Kaggle API (chỉ cần làm 1 lần)
pip install kaggle

# Submit từ terminal
kaggle competitions submit -c house-prices-advanced-regression-techniques \
    -f submission.csv -m "Weighted Ensemble buoi8"
```

---

**Expected Output:**

![img8-8](images/img_buoi8/img8-8.png)



### 📊 Hiểu Kết Quả Kaggle

| Score | Ý nghĩa | Nhận xét |
|-------|---------|---------|
| < 0.11 | Xuất sắc | Top 10–20% leaderboard |
| 0.11 – 0.13 | Tốt | Kết quả bình thường với pipeline cơ bản |
| 0.13 – 0.15 | Trung bình | Cần thêm feature engineering |
| > 0.15 | Cần cải thiện | Kiểm tra lại pipeline |

> **Public Score** = RMSE trên ~50% test set (Kaggle giữ lại 50% để chống gian lận).  
> **Private Score** = RMSE trên 100% test set → chỉ hiện sau khi competition kết thúc.

---

### 💡 Cách Cải Thiện Điểm Số

| Kỹ thuật | Độ khó | Cải thiện dự kiến |
|---------|--------|-------------------|
| Thêm features tương tác (Qual × SF) | Dễ | ~0.003–0.005 |
| Target encoding cho Neighborhood | Trung bình | ~0.002–0.004 |
| Tăng `n_estimators` XGBoost (1000+) | Dễ | ~0.001–0.003 |
| LightGBM thay thế/bổ sung XGBoost | Trung bình | ~0.002–0.005 |
| GridSearchCV đầy đủ | Trung bình | ~0.003–0.008 |
| Loại thêm outlier thông minh | Khó | ~0.001–0.003 |
| Optuna auto-tune hyperparameters | Khó | ~0.005–0.015 |


In [ ]:
# ============================================================
# 🎨 BONUS — Dashboard Tổng Kết Toàn Bộ Buổi 8
# ============================================================
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# ── Panel 1 (top-left): CV RMSE tất cả models (lollipop) ─
ax1 = fig.add_subplot(gs[0, 0])
sorted_results = sorted(results.items(), key=lambda x: x[1])
names_s = [k for k, _ in sorted_results]
rmses_s = [v for _, v in sorted_results]
colors_lol = ['#27ae60' if v == min(rmses_s) else '#3498db' for v in rmses_s]
for i, (nm, rv, col) in enumerate(zip(names_s, rmses_s, colors_lol)):
    ax1.plot([0, rv], [i, i], color='#bdc3c7', lw=2, zorder=1)
    ax1.scatter(rv, i, s=200, c=col, zorder=3, edgecolors='white', lw=1.5)
    ax1.text(rv + 0.0005, i, f'{rv:.4f}', va='center', fontsize=9, fontweight='bold')
ax1.set_yticks(range(len(names_s)))
ax1.set_yticklabels(names_s, fontsize=10)
ax1.set_xlabel('CV RMSE', fontsize=9)
ax1.set_title('CV RMSE Các Models\n(thấp = tốt hơn)', fontweight='bold', fontsize=10)
ax1.axvline(min(rmses_s), color='#27ae60', linestyle=':', alpha=0.5)
ax1.grid(True, axis='x', alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Panel 2 (top-mid): Trọng số ensemble ─────────────────
ax2 = fig.add_subplot(gs[0, 1])
colors_w = ['#3498db','#e74c3c','#27ae60','#9b59b6','#e67e22'][:len(name_list)]
wedges, texts, autotexts = ax2.pie(
    weights, labels=name_list, autopct='%1.1f%%',
    colors=colors_w, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
ax2.set_title('Trọng Số Weighted Ensemble\n(lớn hơn = model tốt hơn)', fontweight='bold', fontsize=10)

# ── Panel 3 (top-right): Histogram dự đoán final ─────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(pred_final / 1000, bins=40, color='#8e44ad', alpha=0.8, edgecolor='white')
ax3.axvline(np.median(pred_final) / 1000, color='#2ecc71', lw=2, linestyle='--',
            label=f'Median ${np.median(pred_final)/1000:.0f}K')
ax3.axvline(pred_final.mean() / 1000,    color='#e67e22', lw=2, linestyle='--',
            label=f'Mean ${pred_final.mean()/1000:.0f}K')
ax3.set_xlabel('Giá dự đoán (nghìn USD)', fontsize=9)
ax3.set_ylabel('Số nhà', fontsize=9)
ax3.set_title('Phân Phối Giá Dự Đoán\n(test set)', fontweight='bold', fontsize=10)
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# ── Panel 4 (bottom, full width): Train vs Test side-by-side
ax4 = fig.add_subplot(gs[1, :])
bins_shared = np.linspace(
    min(train_prices.min(), pred_final.min()) / 1000,
    max(train_prices.max(), pred_final.max()) / 1000,
    60
)
ax4.hist(train_prices / 1000, bins=bins_shared, alpha=0.55,
         color='#2980b9', label=f'Train SalePrice (n={len(train_prices)})', density=True)
ax4.hist(pred_final / 1000,   bins=bins_shared, alpha=0.55,
         color='#e67e22', label=f'Test Predictions (n={len(pred_final)})', density=True)
ax4.set_xlabel('SalePrice (nghìn USD)', fontsize=10)
ax4.set_ylabel('Mật độ (density)', fontsize=10)
ax4.set_title('Train Distribution vs Test Predictions — Hai đường nên "trùng nhau" để đảm bảo model không bị bias',
              fontweight='bold', fontsize=11)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

fig.suptitle('📊 Dashboard Tổng Kết Buổi 8 — Submission Ready!',
             fontsize=14, fontweight='bold', y=1.01)
plt.show()

print(f"\n🏁 TÓM TẮT KẾT QUẢ:")
print(f"   Best CV RMSE     : {min(results.values()):.4f}  ({min(results, key=results.get)})")
print(f"   Ensemble RMSE ước tính: {sum(w * r for w, r in zip(weights, rmse_list)):.4f}")
print(f"   File submission  : {output_path}")
print(f"   Số dự đoán       : {len(submission)}")
print(f"   Giá TB dự đoán   : ${pred_final.mean():,.0f}")
print(f"\n🚀 Sẵn sàng nộp lên Kaggle!")


**Expected Output:**

![img8-9](images/img_buoi8/img8-9.png)
![img8-10](images/img_buoi8/img8-10.png)

---

## 🏁 Tổng Kết Buổi 8

| Task | Nội dung | Kết quả |
|------|---------|---------|
| **1. Rebuild Pipeline** | Load data, fill missing, feature engineering, encode, scale | ✅ Self-contained |
| **2. Train Models** | Ridge, Lasso, RF, XGBoost, Stacking với CV 5-fold | ✅ CV RMSE đánh giá khách quan |
| **3. Weighted Ensemble** | Blend theo trọng số 1/RMSE | ✅ Tốt hơn model đơn lẻ |
| **4. Phân tích dự đoán** | Histogram, boxplot, so sánh Train vs Test distribution | ✅ Kiểm tra chất lượng |
| **5. Tạo CSV** | `submission.csv` đúng format Kaggle | ✅ Sẵn sàng upload |
| **6. Validate** | Kiểm tra tự động 8 tiêu chí | ✅ Tất cả pass |
| **7. Hướng dẫn nộp** | Upload thủ công + Kaggle API | ✅ Xem hướng dẫn Task 7 |

---

### ➡️ Buổi 9 — Quiz Tốt Nghiệp (20 câu ôn tập)

Buổi cuối sẽ ôn lại **toàn bộ kiến thức 9 buổi**:
- 20 câu hỏi covering EDA → Feature Engineering → Modeling → Ensemble
- Lý thuyết + thực hành code
- Giải thích chi tiết từng đáp án

> 🎓 **Chúc mừng!** Bạn đã hoàn thành pipeline đầy đủ: **Raw Data → Kaggle Submission!**
